
Objective:
Build a machine learning regression model to predict the freight
posted_rate using historical shipment data.

Final Deliverables:
1. validation_predictions.csv
2. december_predictions.csv
3. candidate_december.png


##Step 1: Install Required Libraries
Install all required libraries.

CatBoost:
    Main regression model.

LightGBM & XGBoost:
    Installed for experimentation if needed.

Scikit-learn:
    Data preprocessing and evaluation.

Pandas & NumPy:
    Data manipulation.

In [1]:
!pip install catboost lightgbm xgboost

##Step 2: Import Libraries
Import all libraries required for data analysis,
feature engineering, model building and evaluation.


In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

##Step 3: Load Dataset
Load all datasets.

train-test.csv
    Historical labeled data.

validation.csv
    Data used for final prediction.

validation-predictions-template.csv
    Required submission format.

december-chart-inputs.csv
    Fixed December scenarios for visualization.

In [3]:
train = pd.read_csv("/content/train-test.csv")
validation = pd.read_csv("/content/validation.csv")
template = pd.read_csv("/content/validation-predictions-template.csv")
print(template.shape)
template.head()
december = pd.read_csv("/content/december-chart-inputs.csv")

(12000, 2)


##Step 4: Explore Dataset
Understand dataset structure.

Check:
- Number of rows
- Number of columns
- Sample records

In [4]:
print(train.shape)
print(validation.shape)

train.head()

(48000, 14)
(12000, 13)


,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,2025-01-01,0.95684,2.39595,645.41
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,Reefer,17555.0,2025-01-01,0.97623,2.43355,679.97
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,Dry Van,31721.0,2025-01-01,1.00971,1.84491,1802.54
3,TR-000004,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,Dry Van,32333.0,2025-01-01,0.94518,1.87712,1827.28
4,TR-000005,Dallas,Nashville,31.83025,-94.38343,35.29479,-88.08915,541.9,Reefer,35183.0,2025-01-01,0.98480,2.56300,1380.28


##Step 5: Check Data Types
Inspect data types.

Useful for identifying:
- Numerical features
- Categorical features
- Date columns

In [5]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48000 entries, 0 to 47999
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   load_id       48000 non-null  object 
 1   pickup        48000 non-null  object 
 2   delivery      48000 non-null  object 
 3   pickup_lat    48000 non-null  float64
 4   pickup_lon    48000 non-null  float64
 5   delivery_lat  48000 non-null  float64
 6   delivery_lon  48000 non-null  float64
 7   distance      48000 non-null  float64
 8   equipment     48000 non-null  object 
 9   weight        47700 non-null  float64
 10  date          48000 non-null  object 
 11  market_index  47626 non-null  float64
 12  quote_signal  48000 non-null  float64
 13  posted_rate   48000 non-null  float64
dtypes: float64(9), object(5)
memory usage: 5.1+ MB


##Step 6: Check Missing Values
Identify missing values.

Weight and Market Index contain
missing observations which need
to be handled before training.

In [6]:
train.isnull().sum()

,0
load_id,0
pickup,0
delivery,0
pickup_lat,0
pickup_lon,0
delivery_lat,0
delivery_lon,0
distance,0
equipment,0
weight,300


##Step 7: Convert Date Column
Convert date from string format
into datetime.

This enables extraction of
useful calendar-based features.

In [8]:
train["date"] = pd.to_datetime(train["date"])
validation["date"] = pd.to_datetime(validation["date"])
december["date"] = pd.to_datetime(december["date"])

##Step 8: Feature Engineering
Extract useful features from date.

Generated Features:
- Year
- Month
- Day
- Day of Week
- Week Number

These features help the model
capture seasonal trends.

In [9]:
for df in [train, validation, december]:
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["day_of_week"] = df["date"].dt.dayofweek
    df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)

##Step 9: Handle Missing Values
Fill missing values using median.

Median is preferred because
it is less affected by outliers
than the mean.

In [10]:
train["weight"] = train["weight"].fillna(train["weight"].median())
validation["weight"] = validation["weight"].fillna(train["weight"].median())
december["weight"] = december["weight"].fillna(train["weight"].median())

train["market_index"] = train["market_index"].fillna(train["market_index"].median())
validation["market_index"] = validation["market_index"].fillna(train["market_index"].median())

In [11]:
train.isnull().sum()

,0
load_id,0
pickup,0
delivery,0
pickup_lat,0
pickup_lon,0
delivery_lat,0
delivery_lon,0
distance,0
equipment,0
weight,0


##Step 10: Prepare Features
Separate input features (X)
from target variable (y).

Target Variable:
posted_rate

In [12]:
X = train.drop(columns=["posted_rate", "load_id", "date"])
y = train["posted_rate"]

X_test = validation.drop(columns=["load_id", "date"])

##Step 11: Define Categorical Features
CatBoost can directly process
categorical features.

No One-Hot Encoding required.

In [13]:
cat_features = ["pickup", "delivery", "equipment"]

##Step 12: Train Validation Split
Split historical data.

80% → Training

20% → Validation

Random State = 42 ensures
reproducible results.

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

##Step 13: Build CatBoost Model
Train CatBoost Regressor.

Reasons for choosing CatBoost:

• Excellent for tabular datasets

• Handles categorical variables directly

• Handles missing values efficiently

• Reduces preprocessing effort

• Provides strong regression performance

In [15]:
from catboost import CatBoostRegressor

model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42,
    verbose=100
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_features,
    eval_set=(X_valid, y_valid),
    use_best_model=True
)

0:	learn: 1435.2918993	test: 1404.1079436	best: 1404.1079436 (0)	total: 206ms	remaining: 3m 26s
100:	learn: 597.8522812	test: 528.0621465	best: 528.0621465 (100)	total: 19.3s	remaining: 2m 51s
200:	learn: 585.8587523	test: 527.2013954	best: 527.1731100 (168)	total: 25.7s	remaining: 1m 42s
300:	learn: 577.1202091	test: 527.5264308	best: 527.1552636 (217)	total: 30.7s	remaining: 1m 11s
400:	learn: 569.3062631	test: 528.5648455	best: 527.1552636 (217)	total: 37.2s	remaining: 55.6s
500:	learn: 560.1156920	test: 529.0613617	best: 527.1552636 (217)	total: 42.6s	remaining: 42.4s
600:	learn: 551.1672673	test: 529.7751809	best: 527.1552636 (217)	total: 49.6s	remaining: 32.9s
700:	learn: 545.0245162	test: 530.3387657	best: 527.1552636 (217)	total: 54.6s	remaining: 23.3s
800:	learn: 537.2148530	test: 530.8172103	best: 527.1552636 (217)	total: 1m	remaining: 15.1s
900:	learn: 527.4906363	test: 532.3793108	best: 527.1552636 (217)	total: 1m 6s	remaining: 7.35s
999:	learn: 519.7386675	test: 533.521729

CatBoostRegressor(depth=8, eval_metric='RMSE', iterations=1000, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=100)

##Step 14: Model Evaluation
Evaluate model performance.

Metrics Used:

MAE
RMSE
R² Score

These metrics indicate
prediction accuracy.

In [16]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

valid_pred = model.predict(X_valid)

mae = mean_absolute_error(y_valid, valid_pred)
rmse = np.sqrt(mean_squared_error(y_valid, valid_pred))
r2 = r2_score(y_valid, valid_pred)

print("MAE :", mae)
print("RMSE:", rmse)
print("R2 :", r2)

MAE : 102.26761930916116
RMSE: 527.1552588030206
R2 : 0.8700606287772361


##Step 15: Generate Validation Predictions
Predict freight rates for
the unseen validation dataset.

In [17]:
validation_predictions = model.predict(X_test)

In [18]:
print(template.shape)
print(validation.shape)
print(december.shape)

(12000, 2)
(12000, 18)
(31, 12)


##Step 16: Create Submission File
Create submission file using
the required Spotter format.

Columns:
load_id
predicted_rate

In [20]:
submission = template.copy()

submission["predicted_rate"] = validation_predictions

submission.head()

,load_id,predicted_rate
0,TE-000001,879.395864
1,TE-000002,5108.807262
2,TE-000003,5354.785947
3,TE-000004,4322.194543
4,TE-000005,1898.492304


In [21]:
submission.to_csv("validation_predictions.csv", index=False)

print("validation_predictions.csv created successfully")

validation_predictions.csv created successfully


##Step 17: December Prediction
Prepare December dataset by
adding the missing features
required by the trained model.

Generate freight rate predictions
for every day in December.

In [22]:
december_model = december.copy()

december_model = december_model.drop(columns=["date"])

In [23]:
template = pd.read_csv("/content/validation-predictions-template.csv")

submission = template.copy()
submission["predicted_rate"] = validation_predictions

submission.to_csv("validation_predictions.csv", index=False)

print(submission.head())

     load_id  predicted_rate
0  TE-000001      879.395864
1  TE-000002     5108.807262
2  TE-000003     5354.785947
3  TE-000004     4322.194543
4  TE-000005     1898.492304


In [24]:
print(december.columns)

Index(['pickup', 'delivery', 'distance', 'equipment', 'weight', 'date',
       'predicted_rate', 'year', 'month', 'day', 'day_of_week',
       'week_of_year'],
      dtype='object')


In [25]:
print(train.columns.tolist())
print(validation.columns.tolist())

['load_id', 'pickup', 'delivery', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight', 'date', 'market_index', 'quote_signal', 'posted_rate', 'year', 'month', 'day', 'day_of_week', 'week_of_year']
['load_id', 'pickup', 'delivery', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight', 'date', 'market_index', 'quote_signal', 'year', 'month', 'day', 'day_of_week', 'week_of_year']


In [26]:
pickup_coords = (
    train[["pickup", "pickup_lat", "pickup_lon"]]
    .drop_duplicates("pickup")
)

delivery_coords = (
    train[["delivery", "delivery_lat", "delivery_lon"]]
    .drop_duplicates("delivery")
)

In [27]:
december = december.merge(
    pickup_coords,
    on="pickup",
    how="left"
)

december = december.merge(
    delivery_coords,
    on="delivery",
    how="left"
)

In [28]:
december["market_index"] = train["market_index"].median()
december["quote_signal"] = train["quote_signal"].median()

In [29]:
december["date"] = pd.to_datetime(december["date"])

december["year"] = december["date"].dt.year
december["month"] = december["date"].dt.month
december["day"] = december["date"].dt.day
december["day_of_week"] = december["date"].dt.dayofweek
december["week_of_year"] = (
    december["date"].dt.isocalendar().week.astype(int)
)

In [30]:
december_model = december[
    X.columns
]

In [31]:
december["predicted_rate"] = model.predict(
    december_model
)

In [32]:
final_december = december[
    [
        "pickup",
        "delivery",
        "distance",
        "equipment",
        "weight",
        "date",
        "predicted_rate",
    ]
]

final_december.to_csv(
    "december_predictions.csv",
    index=False
)

In [33]:
print(X.columns.tolist())
print(december_model.columns.tolist())

['pickup', 'delivery', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight', 'market_index', 'quote_signal', 'year', 'month', 'day', 'day_of_week', 'week_of_year']
['pickup', 'delivery', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight', 'market_index', 'quote_signal', 'year', 'month', 'day', 'day_of_week', 'week_of_year']


In [34]:
december_predictions = model.predict(december_model)

december["predicted_rate"] = december_predictions

In [35]:
december.to_csv(
    "december_predictions.csv",
    index=False
)

print("December file created")

December file created


In [36]:
from google.colab import files

files.download("validation_predictions.csv")
files.download("december_predictions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [37]:
import os
print(os.listdir("/content"))

['.config', 'catboost_info', 'validation-predictions-template.csv', 'validation.csv', 'validation_predictions.csv', 'train-test.csv', 'december_predictions.csv', 'december-chart-inputs.csv', 'score.py', 'sample_data']


In [46]:
%%writefile score.py

Overwriting score.py


In [47]:
import os
print("score.py" in os.listdir("/content"))

True


In [52]:
import os

print(os.path.exists("score.py"))
print(os.path.exists("validation_predictions.csv"))
print(os.path.exists("december_predictions.csv"))

True
True
True


In [53]:
final_december = december[
    [
        "pickup",
        "delivery",
        "distance",
        "equipment",
        "weight",
        "date",
        "predicted_rate"
    ]
]

print(final_december.columns)
print(final_december.shape)

final_december.to_csv(
    "december_predictions.csv",
    index=False
)

Index(['pickup', 'delivery', 'distance', 'equipment', 'weight', 'date',
       'predicted_rate'],
      dtype='object')
(31, 7)


##Step 18: Validation
Run score.py.

This validates:

✓ File format

✓ Number of rows

✓ Column order

✓ Positive predictions

✓ December prediction format

In [54]:
!python score.py \
--predictions validation_predictions.csv \
--december-predictions december_predictions.csv \
--output-dir scorer_results

Validated 12,000 final predictions.
Validated 31 fixed December predictions.
Created chart: scorer_results/candidate_december.png
Final validation metrics are calculated by Spotter after submission.


In [55]:
from google.colab import files

files.download("scorer_results/candidate_december.png")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>